**Imports**

In [2]:
import matplotlib.pyplot as plt 
import numpy as np
import pandas as pd
import json
import requests
from pathlib import Path
from time import sleep
import io, zipfile
from bs4 import BeautifulSoup as bs
import pyarrow.parquet as pq

**Pandas**

Leer archivo

In [ ]:
pd.read_excel(url, skiprows = 9) #se salta 9 lineas (es ejemplo)
pd.read_csv(archivo_csv)

Transformar a pandas

In [ ]:
archivo.to_pandas()

**JSON**

Abrir un archivo JSON (ej:restaurants.json)

In [ ]:
restaurantes = json.load(open('restaurants.json','r'))

# archivo grande, con encoding
datos_raw = json.load(open('restaurants.json','r', encoding='UTF-8'))

*Tenemos un archivo que se llama json_list*

In [7]:
json_list = [
    { 
        'class': 'Year 1', 
        'student count': 35, 
        'room': 'A2',
        'info': {
            'teachers': { 
                'math': 'Emmy Noether', 
                'physics': 'Richard Feynman' 
            }
        },
        'students': [
            { 
                'name': 'Mary', 
                'sex': 'F', 
                'grades': { 'math': 75, 'physics': 98 } 
            },
            { 
                'name': 'James', 
                'sex': 'M', 
                'grades': { 'math': 80, 'physics': 78 } 
            },
        ]
    },
    { 
        'class': 'Year 2', 
        'student count': 28, 
        'room': 'A4',
        'info': {
            'teachers': { 
                'math': 'Alan Turing', 
                'physics': 'Vera Rubin' 
            }
        },
        'students': [
            { 'name': 'Tony', 'sex': 'M' },
            { 'name': 'Jacqueline', 'sex': 'F' },
        ]
    },
]

Ver los nombres de las columnas

In [8]:
json_list[0].keys()

dict_keys(['class', 'student count', 'room', 'info', 'students'])

Transformar a df con pandas

In [15]:
df = pd.DataFrame(json_list)
df.head()

,class,student count,room,info,students
0,Year 1,35,A2,"{'teachers': {'math': 'Emmy Noether', 'physics...","[{'name': 'Mary', 'sex': 'F', 'grades': {'math..."
1,Year 2,28,A4,"{'teachers': {'math': 'Alan Turing', 'physics'...","[{'name': 'Tony', 'sex': 'M'}, {'name': 'Jacqu..."


Transformar a df más limpio, students es una lista anidada, y debemos agregar class, student count, room e info

In [11]:
all_students_grades= pd.json_normalize(json_list, sep="_", record_path=['students'], meta=['class','student count','room',['info','teachers','math'], ['info','teachers','physics']], max_level=None)
all_students_grades.head()

,name,sex,grades_math,grades_physics,class,student count,room,info_teachers_math,info_teachers_physics
0,Mary,F,75.0,98.0,Year 1,35,A2,Emmy Noether,Richard Feynman
1,James,M,80.0,78.0,Year 1,35,A2,Emmy Noether,Richard Feynman
2,Tony,M,NaN,NaN,Year 2,28,A4,Alan Turing,Vera Rubin
3,Jacqueline,F,NaN,NaN,Year 2,28,A4,Alan Turing,Vera Rubin


In [ ]:
#Otro ejemplo de desanidar
concert_data = pd.json_normalize(data=datos_raw['programs'], record_path='concerts', meta=['id', 'orchestra','programID', 'season'])

Almacenar en archivo json de esta forma

In [13]:
all_students_grades.to_json('students.json')

**REQUESTS**

Tenemos un url `https://raw.githubusercontent.com/fivethirtyeight/data/master/college-majors/recent-grads.csv`


descargar

In [17]:
csv_url = "https://raw.githubusercontent.com/fivethirtyeight/data/master/college-majors/recent-grads.csv"

Guardar

In [18]:
out_dir = Path("data")
out_dir.mkdir(exist_ok=True)
csv_path = out_dir / "recent-grads.csv"

Solicitar datos

In [19]:
r = requests.get(csv_url, timeout=30)
r.raise_for_status()  # Lanza error si el status no es 200
csv_path.write_bytes(r.content)

26872

Ver donde se guardo y que tamaño ocupa

In [20]:
print("Guardado en:", csv_path.resolve())
print("Tamaño (bytes):", csv_path.stat().st_size)

Guardado en: C:\Users\Equipo\AppData\Local\Programs\Microsoft VS Code\data\recent-grads.csv
Tamaño (bytes): 26872


Transformar a df

In [ ]:
df_csv = pd.read_csv(csv_path)
df_csv.head(¿)

,Rank,Major_code,Major,Total,Men,Women,Major_category,ShareWomen,Sample_size,Employed,...,Part_time,Full_time_year_round,Unemployed,Unemployment_rate,Median,P25th,P75th,College_jobs,Non_college_jobs,Low_wage_jobs
0,1,2419,PETROLEUM ENGINEERING,2339.0,2057.0,282.0,Engineering,0.120564,36,1976,...,270,1207,37,0.018381,110000,95000,125000,1534,364,193
1,2,2416,MINING AND MINERAL ENGINEERING,756.0,679.0,77.0,Engineering,0.101852,7,640,...,170,388,85,0.117241,75000,55000,90000,350,257,50
2,3,2415,METALLURGICAL ENGINEERING,856.0,725.0,131.0,Engineering,0.153037,3,648,...,133,340,16,0.024096,73000,50000,105000,456,176,0
3,4,2417,NAVAL ARCHITECTURE AND MARINE ENGINEERING,1258.0,1123.0,135.0,Engineering,0.107313,16,758,...,150,692,40,0.050125,70000,43000,80000,529,102,0
4,5,2405,CHEMICAL ENGINEERING,32260.0,21239.0,11021.0,Engineering,0.341631,289,25694,...,5180,16697,1672,0.061098,65000,50000,75000,18314,4440,972


Manejo de errores

In [22]:
bad_url = "https://api.github.com/this/endpoint/does/not/exist"

try:
    r = requests.get(bad_url, timeout=5)
    r.raise_for_status()
except requests.exceptions.HTTPError as e:
    print("HTTPError:", e)
except requests.exceptions.Timeout:
    print("Timeout alcanzado.")
except requests.exceptions.RequestException as e:
    print("Error de red:", e)

HTTPError: 404 Client Error: Not Found for url: https://api.github.com/this/endpoint/does/not/exist


Otra forma de requests

In [25]:
url = 'https://datos.gob.cl/dataset/c2969d8a-df82-4a6c-a1f8-e5eba36af6cf/resource/cbd329c6-9fe6-4dc1-91e3-a99689fd0254/download/pcma_20240917-oficio-4770_2013.xlsx'
respuesta = requests.get(url,stream=False)

open('puntosBip.xlsx', 'wb').write(respuesta.content) #Escribir respuesta en un archivo llamado puntosBip(ej)

#leemos el archivo local usando pandas
df = pd.read_excel('puntosBip.xlsx', engine='openpyxl',skiprows=9)
df

,CODIGO,ENTIDAD,NOMBRE DE FANTASIA,DIRECCIÓN,COMUNA,HORARIO REFERENCIAL,ESTE,NORTE,LONGITUD,LATITUD
0,1224,Fullcarga,MARION,LOS DURAZNOS 356 A,ESTACION CENTRAL,NaN,342846,6291414,-70.710197,-33.466599
1,3316,Fullcarga,NISSY,JOAQUIN EDWARDS BELLO 10999,LA PINTANA,NaN,349448,6285070,-70.622688,-33.563601
2,9878,Fullcarga,ERICHOPER,LOS RETAMOS 5615,HUECHURABA,NaN,339133,6307750,-70.623108,-33.377819
3,10347,Fullcarga,DON ORLANDO,ARTURO PRAT 5725 DPTO 11,RENCA,NaN,338900,6302656,-70.732925,-33.403622
4,10489,Fullcarga,NOLBERTO VALDES NAVARRO,CALLE 462 5994,PEÑALOLEN,NaN,354367,6292018,-70.568390,-33.501755
...,...,...,...,...,...,...,...,...,...,...
1574,124425,Fullcarga,GINO ANTONY CABRERA LARA,RIO ARHUELLES 3870,PUENTE ALTO,NaN,349313,6279537,-70.624313,-33.613178
1575,124426,Fullcarga,SERVICIOS INTEGRALES LTDA,ESQUINA BLANCA 0261,MAIPU,NaN,337008,6290673,-70.754868,-33.510967
1576,124428,Fullcarga,TODO FARMA 390,PASEO ESTADO 390,SANTIAGO,NaN,346580,6298855,-70.650444,-33.438625
1577,124436,Fullcarga,LA TIENDITA MAGICA,SALAR DE ACOSTAN 10398,LA FLORIDA,NaN,351106,6286304,-70.603859,-33.552422


Descomprimir **ZIP**

In [ ]:
url ='https://www.ine.gob.cl/docs/default-source/geodatos-abiertos/cartografia/censo-2017/siedu/shp/microdatos_manzana.zip?sfvrsn=972b0c54_3'
respuesta = requests.get(url, stream=True)

# Este dataset está en un archivo binario comprimido. Lo abrimos primero como un objeto tipo byte usando la librería io
archivoZip = zipfile.ZipFile(io.BytesIO(respuesta.content))

# Luego descomprimimos usando zipfile
archivoZip.extractall()

%ls Censo2017_16R_ManzanaEntidad_CSV # ver la ruta

dat_manz = pd.read_csv('Censo2017_16R_ManzanaEntidad_CSV/Censo2017_Manzanas.csv',delimiter=';') #abrir con ps en este caso es formato csv
print(dat_manz.head())

Mas sobre requests, sessions, headers en clase 5, 6 y 7(API)

**BeautifulSoup**    mas info clase 8

Leer el archivo html

In [ ]:
html_text=open('ejemplo.html','r').read()
html_text

abrir con bs

In [ ]:
soup = bs(html_text, "html.parser")

soup.table.find_all('tr') # encontrar cosas (ej:tr)
table = soup.find('table') 

Hacer df con bs desde el html

In [ ]:
df = pd.DataFrame(columns=['Curso', 'Creditos', 'nEstudiantes'])
table = soup.find('table')
rows = table.find_all('tr')

for row in rows[1::]:
    cols = row.find_all("td")
    col_text=[c.text for c in cols]
    print(col_text)
    new_row = pd.DataFrame({'Curso':col_text[0], 'Creditos':col_text[1], 'nEstudiantes':col_text[2]}, index=['Curso'])
    df = pd.concat([df, new_row], ignore_index=True)
df

Ejemplo de creación de df con bs y requests

In [32]:
url = "https://en.wikipedia.org/wiki/2023_Rugby_World_Cup_squads"
headers = {"User-Agent": "imt2200-class-notebook"}
page = requests.get(url, headers=headers, timeout=10).text
soup = bs(page)
tables = soup.find_all('table')
# procesar la tabla 20 que tiene la escuadra de Chile
table = tables[19]

#creamos un DataFrame vacío con los títulos de la tabla
df = pd.DataFrame(columns = ['jugador','posicion','nacimiento','caps', 'club'])

# iterar sobre cada fila ('tr') para completar la información
for row in table.find_all('tr')[1::]:
    cols = row.find_all("td")
    #print(cols)
    cols = [col.text.strip() for col in cols]
    #print(cols)
    jugador = cols[0]
    posicion = cols[1]
    nacimiento = cols[2]
    caps = cols[3]
    club = cols[4]
    new_row = pd.DataFrame({'jugador': jugador, 'posicion': posicion, 'nacimiento': nacimiento,'caps':caps, 'club':club}, index=['jugador'])
    df = pd.concat([df, new_row], ignore_index=True)
df

,jugador,posicion,nacimiento,caps,club
0,Augusto Böhme,Hooker,(1997-06-11)11 June 1997 (aged 26),22,Selknam
1,Tomás Dussaillant,Hooker,(1990-10-06)6 October 1990 (aged 32),37,Selknam
2,Diego Escobar,Hooker,(2000-04-17)17 April 2000 (aged 23),5,Selknam
3,Javier Carrasco,Prop,(1997-08-24)24 August 1997 (aged 26),19,Selknam
4,Matías Dittus,Prop,(1993-07-16)16 July 1993 (aged 30),21,Périgueux
5,Iñaki Gurruchaga,Prop,(1995-10-13)13 October 1995 (aged 27),11,Selknam
6,Esteban Inostroza,Prop,(1994-01-01)1 January 1994 (aged 29),1,Selknam
7,Vittorio Lastra,Prop,(1996-03-26)26 March 1996 (aged 27),22,Selknam
8,Salvador Lues,Prop,(1999-11-06)6 November 1999 (aged 23),11,Selknam
9,Javier Eissman,Lock,(1997-03-21)21 March 1997 (aged 26),21,Selknam


**Transformación de datos**

Tipos de datos

In [ ]:
df.dtypes
df.info()
df['columna'].describe() # resumen estadistico valores numericos
df['columna'] = df['columna'].astype('nuevo_tipo') # cambiar tipo (category, int, string, float, etc.)

Info

In [ ]:
mean_datos = df['columna_valores_numericos'].mean() # promedio
categorias = df['columna'].unique() # ver categorias / valores unicos

# Cuántos registros están en la lista de categorías deseadas?
cat_simple = ['forwards','backs']
df['posicion'].isin(cat_simple).sum()

# Cambiar contenido de columna
# Mapeo de categorías originales a nuevas categorías
map_cats={'Hooker':'forwards',
          'Prop':'forwards',
          'Lock':'forwards',
          'Back row':'forwards',
          'Scrum-half':'backs',
          'Fly-half':'backs',
          'Centre':'backs',
          'Wing':'backs',
          'Fullback':'backs'}
df['posicion_simple'] = df['posicion'].map(map_cats)

Datos duplicados

In [ ]:
# ver
duplicates = df.duplicated(keep=False)
dups = df[duplicates].sort_values(by='id')
dups

# quitar manteniendo first, last, etc.
df = df.drop_duplicates(keep='first')

Valores nulos

In [ ]:
# ver
nan_df = df['columna'].isna()
nan_df

# ver como df
df.isna()

df = df.replace('*', np.nan) # Reemplazar * por NaN
df.dropna(subset=['col2']) # Quitar filas con NaN en ej:col2 -no sobreescribe, hay que hacer copia-
df2['col2'] = df2['col2'].fillna(value='d') # Llenar los valores NaN con ej:d

# quitar
df.dropna(subset = ['columna'], inplace=True)

**Parquet**

Leer base parquet

In [ ]:
trips = pq.read_table('data\yellow_tripdata_2022-01.parquet') # Leer
trips = trips.to_pandas() # Transformar a pandas

**Limpieza de datos**

Orden

In [ ]:
trips_by_loc = trips[["PULocationID"]].groupby("PULocationID").size()  # Agrupar datos y saber tamaño
trips_by_loc.sort_values() # Ordenar valores
trips[(trips["PULocationID"] == 237) & (trips["passenger_count"] >= 1)] # Filtrar
df = df.drop(columns=['columna_a_eliminar']) # Eliminar columnas

# Seleccionar columnas
keep_cols = ['id','name','host_id','neighbourhood_group_cleansed','neighbourhood', 'minimum_nights', 'latitude','longitude','price','number_of_reviews']
df = df0[keep_cols]

In [ ]:
# seleccionar strings (ej: (aged 26) )
df['edad'] = df['nacimiento'].str.slice(-3,-1)


**Combinación de datasets**

Concatenar

In [ ]:
df = pd.concat(frames, axis=1) # Concatenar hacia el lado
df = pd.concat(frames, axis=0, join='outer').reset_index(drop=True) # Concatenar hacia bajo

Ejemplo con merge

In [8]:
clientes = pd.read_csv('data\\clientes.csv')
clientes

,Unnamed: 0,Rut,Edad
0,0,18001449,17
1,1,18005940,22
2,2,18012693,19
3,3,18025515,38
4,4,18036316,82
...,...,...,...
845,845,23448775,58
846,846,23454563,35
847,847,23460924,44
848,848,23464397,57


In [9]:
clientes.drop(columns='Unnamed: 0',inplace=True)

In [10]:
clientes

,Rut,Edad
0,18001449,17
1,18005940,22
2,18012693,19
3,18025515,38
4,18036316,82
...,...,...
845,23448775,58
846,23454563,35
847,23460924,44
848,23464397,57


In [4]:
compras = pd.read_csv('data\\compras.csv')
compras

,Unnamed: 0,ID_transaccion,Rut,Fecha,Monto,Local
0,0,11468,18001449,12:33.7,76614,L15
1,1,19219,18001449,21:52.8,17547,L15
2,2,17850,18001449,05:46.6,36592,L5
3,3,21089,18001449,12:11.5,88654,L17
4,4,18385,18001449,55:28.4,60884,L15
...,...,...,...,...,...,...
12295,12295,19778,23490891,42:17.8,43573,L6
12296,12296,18275,23490891,32:28.8,44750,L15
12297,12297,15595,23490891,24:46.1,29884,L1
12298,12298,15206,23490891,22:09.2,89404,L10


In [11]:
compras.drop(columns='Unnamed: 0',inplace=True)

In [16]:
compras

,ID_transaccion,Rut,Fecha,Monto,Local
0,11468,18001449,12:33.7,76614,L15
1,19219,18001449,21:52.8,17547,L15
2,17850,18001449,05:46.6,36592,L5
3,21089,18001449,12:11.5,88654,L17
4,18385,18001449,55:28.4,60884,L15
...,...,...,...,...,...
12295,19778,23490891,42:17.8,43573,L6
12296,18275,23490891,32:28.8,44750,L15
12297,15595,23490891,24:46.1,29884,L1
12298,15206,23490891,22:09.2,89404,L10


Conectar los dos datasets

In [17]:
clientes_compras = clientes.merge(compras, how='left', left_on='Rut', right_on='Rut')
clientes_compras

,Rut,Edad,ID_transaccion,Fecha,Monto,Local
0,18001449,17,11468.0,12:33.7,76614.0,L15
1,18001449,17,19219.0,21:52.8,17547.0,L15
2,18001449,17,17850.0,05:46.6,36592.0,L5
3,18001449,17,21089.0,12:11.5,88654.0,L17
4,18001449,17,18385.0,55:28.4,60884.0,L15
...,...,...,...,...,...,...
12296,23490891,77,19778.0,42:17.8,43573.0,L6
12297,23490891,77,18275.0,32:28.8,44750.0,L15
12298,23490891,77,15595.0,24:46.1,29884.0,L1
12299,23490891,77,15206.0,22:09.2,89404.0,L10


In [ ]:
clientes_compras.groupby(by='Rut').agg({'Monto':'sum'}).reset_index() #sumar monto por cliente

,Rut,Monto
0,18001449,1173796.0
1,18005940,756752.0
2,18012693,352972.0
3,18025515,579702.0
4,18036316,360357.0
...,...,...
845,23448775,211176.0
846,23454563,574606.0
847,23460924,999396.0
848,23464397,691532.0


In [19]:
pv = pd.pivot_table(clientes_compras, index='Rut', values='Monto', aggfunc="sum").reset_index()
pv # sumar monto por cliente en pv

,Rut,Monto
0,18001449,1173796.0
1,18005940,756752.0
2,18012693,352972.0
3,18025515,579702.0
4,18036316,360357.0
...,...,...
845,23448775,211176.0
846,23454563,574606.0
847,23460924,999396.0
848,23464397,691532.0


Distribución de montos de venta por edad

In [20]:
merged = compras.merge(clientes, how='inner')
merged

,ID_transaccion,Rut,Fecha,Monto,Local,Edad
0,11468,18001449,12:33.7,76614,L15,17
1,19219,18001449,21:52.8,17547,L15,17
2,17850,18001449,05:46.6,36592,L5,17
3,21089,18001449,12:11.5,88654,L17,17
4,18385,18001449,55:28.4,60884,L15,17
...,...,...,...,...,...,...
12295,19778,23490891,42:17.8,43573,L6,77
12296,18275,23490891,32:28.8,44750,L15,77
12297,15595,23490891,24:46.1,29884,L1,77
12298,15206,23490891,22:09.2,89404,L10,77


In [21]:
clientes_compras.groupby(by=['Edad']).agg({'Monto':['sum','mean']})

Monto              
             sum          mean
Edad                          
16    11411425.0  46577.244898
17     5523371.0  46414.882353
18     4420052.0  46042.208333
19     9332855.0  45974.655172
20     4217324.0  43033.918367
...          ...           ...
82     7277369.0  48194.496689
83    10775105.0  47467.422907
84     8976593.0  48261.252688
85     7506435.0  45219.487952
86     7455609.0  45461.030488

[71 rows x 2 columns]

Distribución de monto de ventas por edad y local

In [22]:
clientes_compras.groupby(by=['Edad','Local']).agg({'Edad':'min','Monto':['sum','mean']})

Edad     Monto              
            min       sum          mean
Edad Local                             
16   L1      16  717569.0  47837.933333
     L10     16  558420.0  62046.666667
     L11     16  522022.0  43501.833333
     L12     16  824762.0  51547.625000
     L13     16  506336.0  36166.857143
...         ...       ...           ...
86   L5      86  299581.0  37447.625000
     L6      86  552715.0  46059.583333
     L7      86  555074.0  61674.888889
     L8      86  307270.0  43895.714286
     L9      86  295468.0  49244.666667

[1207 rows x 3 columns]

Fracción de ventas que representa cada cliente

In [24]:
def as_perc(value, total):
    return value/float(total)

total_ventas = clientes_compras.Monto.sum()
clientes_compras['frac'] = clientes_compras[['Monto']].apply(as_perc,total=total_ventas)

clientes_compras

,Rut,Edad,ID_transaccion,Fecha,Monto,Local,frac
0,18001449,17,11468.0,12:33.7,76614.0,L15,0.000136
1,18001449,17,19219.0,21:52.8,17547.0,L15,0.000031
2,18001449,17,17850.0,05:46.6,36592.0,L5,0.000065
3,18001449,17,21089.0,12:11.5,88654.0,L17,0.000158
4,18001449,17,18385.0,55:28.4,60884.0,L15,0.000108
...,...,...,...,...,...,...,...
12296,23490891,77,19778.0,42:17.8,43573.0,L6,0.000077
12297,23490891,77,18275.0,32:28.8,44750.0,L15,0.000080
12298,23490891,77,15595.0,24:46.1,29884.0,L1,0.000053
12299,23490891,77,15206.0,22:09.2,89404.0,L10,0.000159


In [25]:
clientes_compras.groupby(by='Rut').agg({'frac':'sum'}).reset_index()

,Rut,frac
0,18001449,0.002087
1,18005940,0.001346
2,18012693,0.000628
3,18025515,0.001031
4,18036316,0.000641
...,...,...
845,23448775,0.000375
846,23454563,0.001022
847,23460924,0.001777
848,23464397,0.001230


Más info sobre combinación y graficos en clase 13

**Exploración de datos**